In [1]:
import pandas as pd

df = pd.read_csv("News Classification Labeling - Sheet1.csv")

print(df.head())
print(df.columns)

                                                 URL  \
0  https://asiatimes.com/2025/10/vietnam-airlines...   
1  https://asiatimes.com/2025/10/doj-seizes-15-bi...   
2  https://www.fool.com/investing/2025/10/15/too-...   
3  https://www.fool.com/investing/2025/10/15/inte...   
4  https://www.fool.com/investing/2025/10/15/3-ro...   

                                               Title    Category Group Leader  
0  Vietnam Airlines data leak exposes a crisis of...  Technology          NaN  
1  DOJ seizes $15 billion bitcoin in SE Asia cryp...     Markets          NaN  
2  Think It's Too Late to Buy IonQ? Here's the 1 ...     Markets          NaN  
3  Intel's Crucial Panther Lake Chips Start Produ...  Technology          NaN  
4                3 Robotics Stocks to Buy in October     Markets          NaN  
Index(['URL', 'Title', 'Category', 'Group Leader'], dtype='object')


In [2]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
print(df.columns.tolist())
text = df["Title"].iloc[0]

result = ner(text)

print(result)
df.columns = df.columns.str.strip()

print(df.columns.tolist())

['URL', 'Title', 'Category', 'Group Leader']
[{'entity_group': 'ORG', 'score': 0.9966762, 'word': 'Vietnam Airlines', 'start': 0, 'end': 16}]
['URL', 'Title', 'Category', 'Group Leader']


In [7]:
import pandas as pd
from transformers import pipeline

# ==========================================
# 1. CSV LOAD
# ==========================================

df = pd.read_csv("News Classification Labeling - Sheet1.csv")

print("Original columns:")
print(df.columns.tolist())


# ==========================================
# 2. COLUMN NAMES CLEAN
# ==========================================

df.columns = df.columns.str.strip().str.lower()

print("\nCleaned columns:")
print(df.columns.tolist())


# ==========================================
# 3. CHECK TITLE COLUMN
# ==========================================

if "title" not in df.columns:
    raise ValueError(
        f"'title' column nahi mili. Available columns: {df.columns.tolist()}"
    )

print("\nTotal rows:", len(df))


# ==========================================
# 4. LOAD PRE-TRAINED NER MODEL
# ==========================================

ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)


# ==========================================
# 5. TEST FIRST ROW
# ==========================================

text = str(df["title"].iloc[0])

print("\nFirst title:")
print(text)

result = ner(text)

print("\nNER result:")
for entity in result:
    print(
        entity["word"],
        "->",
        entity["entity_group"]
    )


# ==========================================
# 6. FUNCTION FOR NER
# ==========================================

def extract_ner(text):

    if pd.isna(text):
        return ""

    entities = ner(str(text))

    results = []

    for entity in entities:

        word = entity["word"]
        label = entity["entity_group"]

        results.append(f"{word} ({label})")

    return ", ".join(results)


# ==========================================
# 7. APPLY NER TO ALL 5000 TITLES
# ==========================================

print("\nProcessing titles...")

df["ner"] = df["title"].apply(extract_ner)


# ==========================================
# 8. CHECK RESULT
# ==========================================

print("\nFinal dataset:")
print(df.head(10))


# ==========================================
# 9. SAVE NEW CSV
# ==========================================

df.to_csv(
    "news_with_ner.csv",
    index=False
)

print("\nDONE!")
print("File saved as: news_with_ner.csv")

Original columns:
['URL', 'Title', 'Category', 'Group Leader']

Cleaned columns:
['url', 'title', 'category', 'group leader']

Total rows: 5000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



First title:
Vietnam Airlines data leak exposes a crisis of transparency

NER result:
Vietnam Airlines -> ORG

Processing titles...

Final dataset:
                                                 url  \
0  https://asiatimes.com/2025/10/vietnam-airlines...   
1  https://asiatimes.com/2025/10/doj-seizes-15-bi...   
2  https://www.fool.com/investing/2025/10/15/too-...   
3  https://www.fool.com/investing/2025/10/15/inte...   
4  https://www.fool.com/investing/2025/10/15/3-ro...   
5  https://www.fool.com/investing/2025/10/15/too-...   
6  https://www.ft.com/content/398b7843-5c25-489c-...   
7  https://www.fool.com/investing/2025/10/15/is-i...   
8  https://www.fool.com/investing/2025/10/15/what...   
9  https://www.fool.com/investing/2025/10/15/down...   

                                               title    category group leader  \
0  Vietnam Airlines data leak exposes a crisis of...  Technology          NaN   
1  DOJ seizes $15 billion bitcoin in SE Asia cryp...     Markets        